In [49]:
import folium
import osmnx as ox
import networkx as nx

ds_kho = {
    "Kho Quận 1": (10.7767, 106.7031),
    "Kho Tân Bình": (10.8016, 106.6520),
    "Kho Quận 7": (10.7290, 106.7216)
}

ds_khach = {
    "Khách 1": (10.7900, 106.7100),
    "Khách 2": (10.7800, 106.7200),
    "Khách 3": (10.7700, 106.6800),
    "Khách 4": (10.7850, 106.7300),
    "Khách 5": (10.7650, 106.6900),
    "Khách 6": (10.8100, 106.6600),
    "Khách 7": (10.8150, 106.6400),
    "Khách 8": (10.7350, 106.7150),
    "Khách 9": (10.7200, 106.7300),
    "Khách 10": (10.7500, 106.6200)
}

tam_ban_do = (10.7767, 106.7031)

G = ox.graph_from_point(tam_ban_do, dist=12000, network_type="drive")

def tim_nut_gan_nhat(toa_do):
    vi_do, kinh_do = toa_do
    return ox.distance.nearest_nodes(G, kinh_do, vi_do)

def tinh_khoang_cach_duong(toa_do_1, toa_do_2):
    nut_1 = tim_nut_gan_nhat(toa_do_1)
    nut_2 = tim_nut_gan_nhat(toa_do_2)

    do_dai = nx.shortest_path_length(G, nut_1, nut_2, weight="length")

    return do_dai / 1000

phan_cong = {}

for ten_khach, toa_do_khach in ds_khach.items():
    kho_gan_nhat = min(
        ds_kho,
        key=lambda ten_kho: tinh_khoang_cach_duong(toa_do_khach, ds_kho[ten_kho])
    )

    if kho_gan_nhat not in phan_cong:
        phan_cong[kho_gan_nhat] = {}

    phan_cong[kho_gan_nhat][ten_khach] = toa_do_khach

def tim_khach_gan_nhat(toa_do_hien_tai, nhom_khach):
    ten_gan_nhat = None
    toa_do_gan_nhat = None
    kc_nho_nhat = None

    for ten_khach, toa_do_khach in nhom_khach.items():
        kc = tinh_khoang_cach_duong(toa_do_hien_tai, toa_do_khach)

        if kc_nho_nhat == None or kc < kc_nho_nhat:
            kc_nho_nhat = kc
            ten_gan_nhat = ten_khach
            toa_do_gan_nhat = toa_do_khach

    return ten_gan_nhat, toa_do_gan_nhat

def tao_tuyen_giao_hang(toa_do_kho, nhom_khach):
    khach_chua_giao = nhom_khach.copy()
    tuyen = [("Kho", toa_do_kho)]
    toa_do_hien_tai = toa_do_kho

    while len(khach_chua_giao) > 0:
        ten_khach, toa_do_khach = tim_khach_gan_nhat(
            toa_do_hien_tai,
            khach_chua_giao
        )

        tuyen.append((ten_khach, toa_do_khach))
        toa_do_hien_tai = toa_do_khach

        del khach_chua_giao[ten_khach]

    tuyen.append(("Kho", toa_do_kho))

    return tuyen

tat_ca_tuyen = {}

for ten_kho, toa_do_kho in ds_kho.items():
    nhom_khach = phan_cong.get(ten_kho, {})

    if len(nhom_khach) > 0:
        tat_ca_tuyen[ten_kho] = tao_tuyen_giao_hang(toa_do_kho, nhom_khach)

def lay_duong_di(toa_do_1, toa_do_2):
    nut_1 = tim_nut_gan_nhat(toa_do_1)
    nut_2 = tim_nut_gan_nhat(toa_do_2)

    duong = nx.shortest_path(G, nut_1, nut_2, weight="length")

    toa_do_duong = []

    for nut in duong:
        vi_do = G.nodes[nut]["y"]
        kinh_do = G.nodes[nut]["x"]
        toa_do_duong.append((vi_do, kinh_do))

    return toa_do_duong

m = folium.Map(location=tam_ban_do, zoom_start=12)

mau_tuyen = {
    "Kho Quận 1": "green",
    "Kho Tân Bình": "purple",
    "Kho Quận 7": "red"
}

for ten_kho, toa_do_kho in ds_kho.items():
    folium.Marker(
        location=toa_do_kho,
        popup=ten_kho,
        tooltip=ten_kho,
        icon=folium.Icon(color=mau_tuyen[ten_kho], icon="home")
    ).add_to(m)

for ten_kho, nhom_khach in phan_cong.items():
    for ten_khach, toa_do_khach in nhom_khach.items():
        folium.Marker(location=toa_do_khach, popup=ten_khach + " - " + ten_kho, tooltip=ten_khach).add_to(m)

for ten_kho, tuyen in tat_ca_tuyen.items():
    mau = mau_tuyen[ten_kho]

    for i in range(len(tuyen) - 1):
        toa_do_dau = tuyen[i][1]
        toa_do_cuoi = tuyen[i + 1][1]

        folium.PolyLine(locations=lay_duong_di(toa_do_dau, toa_do_cuoi), color=mau, weight=4, opacity=0.8).add_to(m)

m